# Vector算子开发实战

实现Ascend C算子LogSigmoid，算子命名为LogSigmoidCustom，编写其kernel侧代码、host侧代码，并完成单算子调用测试。
相关算法：
  $$
  LogSigmoid(x) = \log(\frac{1}{1+\exp(-x)})
  $$

要求：

- 完成LogSigmoid算子kernel侧核函数实现相关代码补齐。

- 完成LogSigmoid算子host侧Tiling函数相关代码补齐。

- 要支持float/half/bfloat16类型输入输出。

- 支持多条调用case并运行成功：
    * case1：输入x的shape为(8,16,64)，类型为float32
    * case2：输入x的shape为(8,16,1743)，类型为float32
    * case3：输入x的shape为(4,2028)，类型为float16
    * case4：输入x的shape为(32,1001,7763)，类型为float16
    * case5：输入x的shape为(1,1024)，类型为bfloat16
    * case6：输入x的shape为(64,64,1024)，类型为bfloat16

请开始你的实践，体验完整开发过程。

---

### 环境准备：

In [ ]:
!mkdir -p Sources/test

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("\n🎉 Environment initialization process completed successfully!")

算子工程创建工具msopgen准备：

In [ ]:
# CANN 9.0.0 已自带 msopgen 工具，无需源码编译安装
!which msopgen && msopgen -h

--- 
### 算子原型json文件准备
该部分不需要修改，直接执行即可。

In [ ]:
%%writefile Sources/test/log_sigmoid_custom.json
[{
    "op": "LogSigmoidCustom",
    "input_desc": [{
            "name": "x",
            "param_type": "required",
            "format": ["ND", "ND", "ND"],
            "type": ["float16", "float", "bf16"]
        }
    ],
    "output_desc": [{
        "name": "y",
        "param_type": "required",
        "format": ["ND", "ND", "ND"],
        "type": ["float16", "float", "bf16"]
    }]
}]

---
### 算子工程创建
该部分不需要修改，直接执行即可。

In [ ]:
# 清除已有的custom_op目录
!rm -rf Sources/test/custom_op

# 使用msopgen工具创建算子工程前设置使用开源仓编译的msopgen工具，不设置时默认使用CANN安装目录下的msopgen工具
!export PATH=/usr/local/bin:~/.local/bin:$PATH;msopgen gen -i Sources/test/log_sigmoid_custom.json -c ai_core-ascend910b1 -lan cpp -out Sources/test/custom_op


---
### host侧代码修改
修改代码，完成Tiling实现：

In [ ]:
%%writefile Sources/test/custom_op/op_host/log_sigmoid_custom.cpp
#include "../op_kernel/log_sigmoid_custom_tiling.h"
#include "register/op_def_registry.h"

namespace optiling {
static ge::graphStatus TilingFunc(gert::TilingContext* context)
{

  LogSigmoidCustomTilingData *tiling = context->GetTilingData<LogSigmoidCustomTilingData>();
  const gert::StorageShape* x1_shape = context->GetInputShape(0);
  // 修改Tiling实现，如需引用头文件请自行添加
  int32_t data_sz = 1;
  for (int i = 0; i < x1_shape->GetStorageShape().GetDimNum(); i++)
    data_sz *= x1_shape->GetStorageShape().GetDim(i);
  tiling->size = data_sz;
  context->SetBlockDim(8);
  size_t *currentWorkspace = context->GetWorkspaceSizes(1);
  currentWorkspace[0] = 0;
  return ge::GRAPH_SUCCESS;
}
}

namespace ge {
static ge::graphStatus InferShape(gert::InferShapeContext* context)
{
    const gert::Shape* x1_shape = context->GetInputShape(0);
    gert::Shape* y_shape = context->GetOutputShape(0);
    *y_shape = *x1_shape;
    return GRAPH_SUCCESS;
}
static ge::graphStatus InferDataType(gert::InferDataTypeContext *context)
{
    const auto inputDataType = context->GetInputDataType(0);
    context->SetOutputDataType(0, inputDataType);
    return ge::GRAPH_SUCCESS;
}
}


namespace ops {
class LogSigmoidCustom : public OpDef {
public:
    explicit LogSigmoidCustom(const char* name) : OpDef(name)
    {
        this->Input("x")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT, ge::DT_BF16})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND, ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND, ge::FORMAT_ND, ge::FORMAT_ND});
        this->Output("y")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT, ge::DT_BF16})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND, ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND, ge::FORMAT_ND, ge::FORMAT_ND});

        this->SetInferShape(ge::InferShape).SetInferDataType(ge::InferDataType);

        this->AICore()
            .SetTiling(optiling::TilingFunc);
        this->AICore().AddConfig("ascend910b");

    }
};

OP_ADD(LogSigmoidCustom);
}


---
### Tiling结构体定义修改
修改代码，完成Tiling结构体定义：

In [ ]:
%%writefile  Sources/test/custom_op/op_kernel/log_sigmoid_custom_tiling.h
#ifndef LOG_SIGMOID_CUSTOM_TILING_H
#define LOG_SIGMOID_CUSTOM_TILING_H
#include <cstdint>

struct LogSigmoidCustomTilingData {
    // 修改Tiling结构体定义
    uint32_t size;
};

#endif // LOG_SIGMOID_CUSTOM_TILING_H

---
### Kernel实现修改
修改代码，完成Kernel实现：

In [ ]:
%%writefile  Sources/test/custom_op/op_kernel/log_sigmoid_custom.cpp
#include "kernel_operator.h"
#include "log_sigmoid_custom_tiling.h"

extern "C" __global__ __aicore__ void log_sigmoid_custom(GM_ADDR x, GM_ADDR y, GM_ADDR workspace, GM_ADDR tiling) {
    REGISTER_TILING_DEFAULT(LogSigmoidCustomTilingData);
    GET_TILING_DATA(tilingData, tiling);
    // 请完成Kernel侧代码实现
}

修改完成后，部署算子运行测试，该部分不需要修改，直接执行即可。

In [ ]:
# 编译部署修改后的算子
!cd Sources/test/custom_op;bash build.sh;./build_out/custom_opp*.run --install-path=${HOME}/

---
### 跑通所有测试用例
该部分不需要修改，直接执行即可。

In [ ]:
!bash src/09.01_testcase/testcase_1/run.sh

In [ ]:
!bash src/09.01_testcase/testcase_2/run.sh

In [ ]:
!bash src/09.01_testcase/testcase_3/run.sh

In [ ]:
!bash src/09.01_testcase/testcase_4/run.sh

In [ ]:
!bash src/09.01_testcase/testcase_5/run.sh

In [ ]:
!bash src/09.01_testcase/testcase_6/run.sh